# 4 · JSON and Cypher — the same engine, addressed two other ways

There is one query builder. The Python API, the JSON API and the Cypher subset are
front ends that compile to the same `(Start, [Hop])` pair and the same
`resolve()`-compiled filters, which is why they cannot drift apart.

Why bother: **a model already knows Cypher and JSON tool calls.** Anything invented
here would need a paragraph of prompt to use correctly, and would be got wrong
anyway.

In [1]:
import json

from demo_graph import arrows, connect, names, seed
from hopai import (
    AGGREGATE_TOOL_SCHEMA, INGEST_TOOL_SCHEMA, TRAVERSE_TOOL_SCHEMA, CypherError, Hop, Start,
    Unique, aggregate_json, cypher_to_traversal, spec_to_traversal, traverse_json,
)

graph = connect("nb_04_json_cypher")
seed(graph)
print(names(graph.traverse(Start())))

['Acme', 'Alice', 'Bob', 'Carol', 'Dave', 'Erin', 'Globex']


## One question, three spellings

*"Which companies do Alice's friends — up to four hops out, active only — work
for?"*

In [2]:
python_result = graph.traverse(
    Start(where={"name": "Alice"}),
    Hop(via={"kind": "friend"}, hops=(1, 4), where={"active": True}),
    Hop(via={"kind": "works_at"}, where={"type": "company"}),
)

json_result = traverse_json(graph, {
    "start": {"where": {"name": "Alice"}},
    "hops": [
        {"via": {"kind": "friend"}, "hops": [1, 4], "where": {"active": True}},
        {"via": {"kind": "works_at"}, "where": {"type": "company"}},
    ],
})

cypher_result = graph.cypher("""
    MATCH (a {name: 'Alice'})-[:friend*1..4]->(b {active: true})-[:works_at]->(c:company)
    RETURN c
""")

# traverse_json is JSON in, JSON out: a plain dict, not a Subgraph.
json_names = sorted(n["properties"]["name"] for n in json_result["nodes"])

assert names(python_result) == json_names == names(cypher_result)
print("all three agree on the subgraph:", names(python_result))
print("the companies in it:", [n["properties"]["name"] for n in python_result.nodes
                               if n["properties"]["type"] == "company"])

all three agree on the subgraph: ['Acme', 'Alice', 'Bob', 'Carol', 'Dave', 'Erin']
the companies in it: ['Acme']


## The JSON API

Same keys, same meaning, JSON in and JSON out — for an LLM tool call, an HTTP
handler, or config-driven traversal. Filters take the same grammar spelled as
operators: `{"and": [...]}`, `{"or": [...]}`, `{"not": ...}`, `{"gt": [key, value]}`,
`{"gte": ...}`, `{"lt": ...}`, `{"lte": ...}`, `{"between": [key, lo, hi]}`.

In [3]:
spec = {
    "start": {"where": {"and": [{"type": "person"},
                                {"not": {"city": "Berlin"}},
                                {"gt": ["age", 22]}]}},
}
result = traverse_json(graph, spec)
print(sorted(n["properties"]["name"] for n in result["nodes"]))
print(json.dumps(result["nodes"][0], indent=2))

['Carol', 'Dave', 'Erin']
{
  "id": "3",
  "properties": {
    "age": 29,
    "city": "Lisbon",
    "name": "Carol",
    "type": "person",
    "email": "carol@example.com",
    "active": true
  }
}


`traverse_json` hands back a plain `{"nodes": [...], "edges": [...], "elapsed_ms": ...}`
dict rather than a `Subgraph` — no Python objects in, none out, so the return value
goes straight into a tool result. And the translation is inspectable without
running anything, which is the thing to log when a model writes the spec:

In [4]:
start, hops = spec_to_traversal({
    "start": {"where": {"name": "Alice"}},
    "hops": [{"via": {"kind": "friend"}, "hops": [1, 4], "where": {"gte": ["age", 30]}},
             {"via": {"kind": "works_at"}, "direction": "forward", "where": {"type": "company"}}],
})
print(start)
for hop in hops:
    print(hop)

Start(where={'name': 'Alice'}, label=None)
Hop(where=GTE('age', 30), via={'kind': 'friend'}, hops=(1, 4), direction='forward', optional=False, label=None)
Hop(where={'type': 'company'}, via={'kind': 'works_at'}, hops=1, direction='forward', optional=False, label=None)


### The tool schemas

`TRAVERSE_TOOL_SCHEMA`, `AGGREGATE_TOOL_SCHEMA` and `INGEST_TOOL_SCHEMA` are
ready-made JSON Schemas for a function-calling definition — reading, aggregating
and writing. They are kept in step with what the parsers accept, so what a model is
told it may send is what the parser takes.

In [5]:
print(TRAVERSE_TOOL_SCHEMA["name"], "->", TRAVERSE_TOOL_SCHEMA["description"][:90], "...")
print(json.dumps(TRAVERSE_TOOL_SCHEMA["parameters"]["properties"]["start"], indent=2))
print([s["name"] for s in (TRAVERSE_TOOL_SCHEMA, AGGREGATE_TOOL_SCHEMA, INGEST_TOOL_SCHEMA)])

traverse_graph -> Traverse a property graph stored in PostgreSQL. Follow edges from a starting set of nodes  ...
{
  "type": "object",
  "description": "The seed set of nodes to begin from.",
  "properties": {
    "where": {
      "type": "object",
      "description": "Filter on starting nodes."
    }
  },
  "required": [
    "where"
  ]
}
['traverse_graph', 'aggregate_graph', 'ingest_graph']


Wiring one up is the obvious thing — the schema goes in the tool definition, and
the model's arguments go straight into `traverse_json`:

```python
# Each schema is {"name", "description", "parameters"}. For the Anthropic API,
# `parameters` is the thing to pass as `input_schema`:
tools = [{"name": s["name"], "description": s["description"], "input_schema": s["parameters"]}
         for s in (TRAVERSE_TOOL_SCHEMA, AGGREGATE_TOOL_SCHEMA, INGEST_TOOL_SCHEMA)]
...
if block.name == "traverse_graph":
    result = traverse_json(graph, block.input)      # already JSON-ready
```

### Aggregating, in JSON

In [6]:
aggregate_json(graph, {
    "start": {"where": {"type": "person"}},
    "hops": [{"via": {"kind": "friend"}, "hops": [1, 4]}],
    "aggregates": {"reached": {"fn": "count"},
                   "avg_age": {"fn": "avg", "property": "age"}},
})

{'reached': 5, 'avg_age': 35.8}

### Writing, in JSON

One document, nodes written before edges, so an edge may reference a node the same
document creates.

In [7]:
graph.ingest({
    "nodes": [{"id": 900, "type": "person", "name": "Grace", "active": True},
              {"id": 901, "type": "company", "name": "Initech", "founded": 2004}],
    "edges": [{"start_id": 900, "end_id": 901, "kind": "works_at", "since": 2023}],
})

IngestResult(nodes=2, edges=1, elapsed_ms=4.0)

## Cypher as input syntax

For callers who already think in Cypher — and for a model, which has seen far more
Cypher than it has seen any Python API.

`graph.cypher()` returns a `Subgraph` for a query that reads, an `IngestResult` for
one that writes, and a plain dict for one whose `RETURN` aggregates.

In [8]:
read = graph.cypher("""
    MATCH (a:person {name: 'Alice'})-[:friend*1..2]->(b {active: true})
    WHERE b.age > 25
    RETURN b
""")
print(names(read))
print(arrows(read))

['Alice', 'Bob', 'Carol']
['Alice -friend-> Bob', 'Alice -friend-> Carol']


hopai has no label concept, so **labels compile to property tests**: `(a:person)`
becomes `{"type": "person"}` and `[:friend]` becomes `{"kind": "friend"}`. Change
the keys with `node_label_key=` / `edge_type_key=`, or pass `None` to ignore labels
entirely.

`cypher_to_traversal()` shows the translation without running it — this is the
thing to print when a query answers something you did not expect:

In [9]:
start, hops = cypher_to_traversal("""
    MATCH (a:person {name: 'Alice'})-[:friend*1..2]->(b {active: true})
    WHERE b.age > 25 AND b.city IN ['Berlin', 'Lisbon']
    RETURN b
""")
print(start)
for hop in hops:
    print(hop)

Start(where={'type': 'person', 'name': 'Alice'}, label='a')
Hop(where=AND({'active': True, 'city': ['Berlin', 'Lisbon']}, GT('age', 25)), via={'kind': 'friend'}, hops=(1, 2), direction='forward', optional=False, label='b')


### Aggregating in Cypher, and the one subtlety worth reading

Cypher aggregates over result **rows** — one per path. hopai aggregates over the
distinct nodes the last step matched. Where the two coincide, the query translates
exactly; where they do not, it raises and names the rewrite.

In [10]:
print(graph.cypher("MATCH (a:person)-[:friend*1..4]->(b) RETURN count(DISTINCT b)"))
print(graph.cypher("MATCH (a:person)-[:friend]->(b) WITH DISTINCT b RETURN avg(b.age)"))
print(graph.cypher("MATCH (a:person) RETURN count(a)"))     # no hops: one node, one row

{'count': 5}
{'avg_age': 35.8}
{'count': 6}


In [11]:
try:
    graph.cypher("MATCH (a:person)-[:friend*1..4]->(b) RETURN count(b)")
except CypherError as exc:
    print(f"CypherError: {exc}")

CypherError: bare count(b) aggregates one row per PATH -- a node two paths reach counts twice -- and hopai does not track path multiplicity across hops. Write count(DISTINCT ...) to aggregate distinct values, or `WITH DISTINCT b RETURN count(...)` to aggregate each matched node once


A node two paths reach counts twice in Cypher and once here. Rather than answer a
different question quietly, the translator refuses and names both exact rewrites.
This is the library's central rule — **refuse, don't approximate** — and it is why
you can hand this front end to a model without auditing every answer.

### Writing in Cypher

Writes compile to the same `add_nodes` / `merge_nodes` / `add_edges` the Python API
calls, in one transaction, with ids from the insert wiring the edges.

In [12]:
graph.cypher("""
    CREATE (h:person {name: 'Heidi', email: 'heidi@example.com', active: true})
          -[:friend]->(i:person {name: 'Ivan', email: 'ivan@example.com', active: true})
""")

IngestResult(nodes=2, edges=1, elapsed_ms=2.9)

`MERGE` is the idempotent one — the right call for an agent that might retry. It
needs a **unique index over every property in the pattern**, because those are the
keys it matches on; the error names the `Unique(...)` to declare when there isn't
one. Note that the label counts as a property here (`:person` → `type`):

In [13]:
try:
    graph.cypher("MERGE (a:person {email: 'judy@example.com'}) ON CREATE SET a.name = 'Judy'")
except Exception as exc:
    print(f"{type(exc).__name__}: {exc}")

ConstraintViolation: merge_nodes(on=['email', 'type']) needs a unique index over exactly those keys to detect a conflict, and there is none. Declare it first: graph.define_constraints(nodes=[Unique('email', 'type')])


In [14]:
graph.define_constraints(nodes=[Unique("email", "type")])

for run in (1, 2):
    written = graph.cypher("""
        MERGE (a:person {email: 'judy@example.com'})
        ON CREATE SET a.name = 'Judy'
        ON MATCH SET  a.last_seen = 2026
    """)
    print(f"run {run}: {written}")

print(graph.traverse(Start(where={"email": "judy@example.com"})).nodes)

run 1: IngestResult(nodes=1, edges=0, elapsed_ms=2.3)
run 2: IngestResult(nodes=1, edges=0, elapsed_ms=1.9)
[{'id': '904', 'properties': {'name': 'Judy', 'type': 'person', 'email': 'judy@example.com', 'last_seen': 2026}}]


Run twice, one node — with `last_seen` stamped by the second run. `ON MATCH SET`
merges over the existing properties and leaves the rest alone.

`graph.cypher_operations()` is the dry run: the plan, without touching anything.

In [15]:
for operation in graph.cypher_operations(
    "CREATE (a:person {name: 'Ken'})-[:friend]->(b:person {name: 'Leo'})"
):
    print(operation)

{'op': 'create_nodes', 'rows': [{'type': 'person', 'name': 'Ken'}, {'type': 'person', 'name': 'Leo'}], 'vars': ['a', 'b']}
{'op': 'create_edges', 'rows': [{'start_var': 'a', 'end_var': 'b', 'properties': {'kind': 'friend'}}]}


## What Cypher refuses, and why

Every one of these raises `CypherError` naming the rewrite, rather than translating
into something that answers a different question. Reading the messages is the
fastest way to learn the boundary of the subset:

In [16]:
refused = [
    "MATCH (a:person) WHERE a.city <> 'Berlin' RETURN a",
    "MATCH (a)-[:friend*]->(b) RETURN b",
    "MATCH (a:person)-[:friend]->(b) RETURN b.city, count(DISTINCT b)",
    "MATCH (a:person)-[:friend]->(b) WHERE a.age > 30 OR b.age > 30 RETURN b",
    "MERGE (a:person {email: 'x@example.com'})-[:friend]->(b:person {email: 'y@example.com'})",
    "MATCH (a:person)-[:friend]->(b) RETURN b ORDER BY b.age LIMIT 3",
]
for query in refused:
    try:
        graph.cypher(query)
        print(f"!! translated unexpectedly: {query}")
    except CypherError as exc:
        print(f"{query.strip()[:58]:60} {exc}\n")

MATCH (a:person) WHERE a.city <> 'Berlin' RETURN a           `<>` is not supported on its own. Cypher and hopai disagree here, so this raises rather than translating into something that answers a different question: Cypher evaluates `x.k <> v` to NULL when `k` is missing and drops that row, while hopai's containment-based NOT keeps it. Write the NULL-safe form -- `x.k IS NULL OR x.k <> v` -- which maps exactly onto NOT({'k': v})

MATCH (a)-[:friend*]->(b) RETURN b                           hop 0: an unbounded `*` has no equivalent -- the recursive walk needs an upper bound to terminate. Write `*1..N`, or pass max_var_length=N to cap every unbounded pattern in the query

MATCH (a:person)-[:friend]->(b) RETURN b.city, count(DISTI   RETURN mixes an aggregate with a plain item at position 39 -- that is grouped aggregation (GROUP BY), which hopai does not support yet. Either aggregate every item, or drop the aggregates and take the subgraph

MATCH (a:person)-[:friend]->(b) WHERE a.age > 30 

`SET` on matched rows, `DELETE` and `DETACH DELETE` are unsupported for a simpler
reason: there is no update-by-query or delete API here yet, in Cypher or in Python.

---

Next: [05 · Constraints](05_constraints.ipynb) — the integrity features Neo4j puts
behind an enterprise licence, which Postgres has always had.